# Testing using Alphafold components on Google Colab

In [4]:
#@title Run to install dependencies
#@markdown * Copied from the [Alphafold Colab notebook](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)

from IPython.utils import io
import os
import subprocess
import tqdm.notebook

TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

GIT_REPO = 'https://github.com/deepmind/alphafold'
DATA_REPO = 'https://github.com/sameerd/data_dump'

try:
  with tqdm.notebook.tqdm(total=100, bar_format=TQDM_BAR_FORMAT) as pbar:
    with io.capture_output() as captured:
      # Uninstall default Colab version of TF.
      %shell pip uninstall -y tensorflow
      pbar.update(6)
      %shell rm -rf alphafold
      %shell git clone --branch main {GIT_REPO} alphafold
      pbar.update(8)
      %shell pip3 install -r ./alphafold/requirements.txt
      pbar.update(50)
      # Run setup.py to install only AlphaFold.
      %shell pip3 install --no-dependencies ./alphafold
      pbar.update(32)    
      %shell git clone --branch main {DATA_REPO} data_dump
      pbar.update(4)

except subprocess.CalledProcessError:
  print(captured)
  raise


import jax
if jax.local_devices()[0].platform == 'tpu':
  raise RuntimeError('Colab TPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
elif jax.local_devices()[0].platform == 'cpu':
  #raise RuntimeError('Colab CPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
  print(f"{jax.local_devices()}")
else:
  print(f'Running with {jax.local_devices()[0].device_kind} GPU')


# Make sure all necessary environment variables are set.
import os
os.environ['TF_FORCE_UNIFIED_MEMORY'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '2.0'





  0%|          | 0/100 [elapsed: 00:00 remaining: ?]

[CpuDevice(id=0)]


In [9]:
import dataclasses
import gzip
import alphafold

from alphafold.common.protein import from_pdb_string

### Data input

In [10]:
with gzip.open("./data_dump/starting.pdb.gz", "rt") as fh:
    prot = from_pdb_string(fh.read())
    
dataclasses.fields(prot)

(Field(name='atom_positions',type=<class 'numpy.ndarray'>,default=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),_field_type=_FIELD),
 Field(name='aatype',type=<class 'numpy.ndarray'>,default=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),_field_type=_FIELD),
 Field(name='atom_mask',type=<class 'numpy.ndarray'>,default=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),_field_type=_FIELD),
 Field(name='residue_index',type=<class 'numpy.ndarray'>,default=<dataclasses._MISSING_TYPE object at 0x7fcaa772ad90>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fcaa7

In [11]:
prot.aatype

array([12,  5,  8, 16, 18, 14,  0,  5, 10, 12,  1, 13,  7, 16,  0,  0,  1,
        0,  6,  8, 12, 16,  9,  0,  0,  0,  9,  8,  0, 10,  3,  0,  3,  6,
        0,  3,  0,  9, 19, 12,  3,  9, 19, 14,  3,  7,  6,  1,  3,  0, 17,
       17,  3,  3,  6,  7, 13, 15, 15, 15, 14, 13, 16, 11,  2,  0,  8,  8,
        0,  7,  9, 19,  0, 16, 15, 19, 16, 10,  7,  5, 10,  5,  1,  6,  5,
        7,  3, 11, 10, 19, 15, 11,  0,  0,  6, 18, 13,  7,  9,  0,  4,  1,
       19,  2,  3,  7, 10,  1, 16, 16,  1, 13, 19,  1, 10, 13, 15,  3,  0,
       10,  3,  0, 11, 14, 10, 16,  9,  7,  8,  3, 18,  6, 19,  6, 13, 10,
       10,  0, 16,  1,  1, 19, 18,  6, 14, 13,  6,  0, 14, 13,  2, 13,  0,
       14,  8,  4,  3,  3, 19, 15, 18,  7,  1,  3, 16, 19,  2, 17, 14, 10,
       11,  1, 15, 13, 14,  1,  5, 10,  7,  7, 13, 10, 16,  9,  5,  7,  0,
        3,  2,  3,  0,  7, 12, 19, 12, 17,  3,  2,  1, 14,  6, 15,  1,  0,
        0, 10,  3,  6, 12,  8,  0,  6, 18,  1,  6, 16,  7,  0,  9,  0,  0,
       10,  6,  1,  0,  0

In [12]:
prot.atom_mask

array([[1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       ...,
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 1.]])

In [13]:
prot.atom_positions

array([[[-40.47900009,  18.67099953, -24.95700073],
        [-40.76399994,  18.16799927, -23.57099915],
        [-40.06900024,  16.84399986, -23.38299942],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       [[-39.38000107,  16.73399925, -22.25799942],
        [-38.65000153,  15.54899979, -21.84399986],
        [-39.60800171,  14.58399963, -21.22200012],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       [[-39.36100006,  13.30000019, -21.38500023],
        [-40.18799973,  12.31799984, -20.70899963],
        [-39.91699982,  12.22200012, -19.23999977],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       ...,

      

In [14]:
prot.b_factors

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [15]:
prot.chain_index

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0])

In [16]:
prot.residue_index

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

## Model

In [17]:
from alphafold.model.common_modules import Linear

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk

In [18]:
def _f(x):
    m = Linear(num_output=2, num_input_dims=1, name="linear22")
    return m(x)
f = hk.transform(_f)
f

Transformed(init=<function without_state.<locals>.init_fn at 0x7fca604e55f0>, apply=<function without_state.<locals>.apply_fn at 0x7fca604e5560>)

In [19]:
test_inp = prot.atom_positions.astype(jnp.float32)

In [20]:
rng_key = jax.random.PRNGKey(42)
params = f.init(x=test_inp, rng=rng_key)
params

FlatMapping({
  'linear22': FlatMapping({
                'weights': DeviceArray([[-0.34503725,  0.582498  ],
                                        [ 0.0910517 , -0.35958534],
                                        [ 0.68855035,  0.66172487]], dtype=float32),
                'bias': DeviceArray([0., 0.], dtype=float32),
              }),
})

In [21]:
prot.atom_positions.shape

(273, 37, 3)

In [22]:
output1 = f.apply(params, rng=rng_key, x=test_inp)
output1

DeviceArray([[[ -1.5173626 , -46.807423  ],
              [ -0.51049376, -45.875412  ],
              [ -0.74139994, -44.870083  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[ -0.21452728, -43.684746  ],
              [ -0.28924042, -42.55946   ],
              [  0.38171828, -42.358902  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[  0.0673494 , -41.861176  ],
              [  0.7287432 , -41.54246   ],
              [  1.6379772 , -40.37801   ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             ...,

             [[ -6.134639  , -31.984861  ],
              [ -6.269721  , -31.408993  ],
              [ -6.9840364 , -30.61801